# PharmaRL — GRPO training on SARS-CoV-2 Mpro env

**What this does**: trains Qwen3-1.5B to design Mpro inhibitor candidates by editing SELFIES strings. Talks to a deployed PharmaRL OpenEnv (HF Space) over HTTP. Logs reward curves to W&B.

**Runtime**: Colab T4 (free) is enough thanks to Unsloth 4-bit + LoRA.

**Env URL**: edit `ENV_URL` below to point at your HF Space.

## 1. Install

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-cache-dir 'trl>=0.11.0' 'transformers>=4.40.0' 'peft' 'accelerate'
!pip install -q openenv-core[core] selfies rdkit-pypi PyTDC wandb

## 2. Config

In [ ]:
import os

ENV_URL = os.environ.get('PHARMARL_ENV_URL', 'http://localhost:8000')  # set to HF Space URL once deployed
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LEN = 1024
LORA_RANK = 16
NUM_GENERATIONS = 8           # G in GRPO — must be >= 2
MAX_TRAINING_STEPS = 200          # auditor: realistic on T4 in 8h is 150-300 steps; 500 overshoots
WANDB_PROJECT = 'pharmarl'
WANDB_RUN_NAME = 'multitarget-drd2-gsk3b-heldout-jnk3'

# Held-out generalization test setup:
#   Train: rotate per-episode through TRAINING_TARGETS
#   Eval:  measure on HELD_OUT_TARGET — a target the model NEVER sees during training.
# Compare untrained-vs-trained scores on HELD_OUT_TARGET to test transfer.
TRAINING_TARGETS = ['DRD2', 'GSK3B']
HELD_OUT_TARGET = 'JNK3'
EVAL_EPISODES_PER_TARGET = 8     # how many episodes to average for before/after
EVAL_DIFFICULTY = 'easy'         # binding component is active here (trivial is QED-only)


## 3. Load model with Unsloth (4-bit + LoRA)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

## 4. Connect to env

In [ ]:
import requests
import uuid

# Smoke test: hit the env once. NOTE: episode_id required — the env keeps state across calls per id.
_eid = str(uuid.uuid4())
r = requests.post(f'{ENV_URL}/reset', json={'difficulty': 'trivial', 'episode_id': _eid}).json()
print('reset OK:', r['observation']['smiles'], '   episode_id:', r['observation']['episode_id'])
print('vocab:', r['observation']['available_fragments'])


## 5. Reward functions (GRPO-style — multi-component)

GRPO accepts a list of reward functions; each maps `(completion, env_response)` to a float. We separate them so W&B graphs each component independently.

In [ ]:
import re
import json

_JSON_RE = re.compile(r'\{[^{}]*\}')

def parse_action(text: str):
    for m in _JSON_RE.findall(text):
        try:
            return json.loads(m)
        except Exception:
            continue
    return None

def reward_format(prompts, completions, **kwargs):
    """Was the model output a parseable JSON action?"""
    return [0.5 if parse_action(c) is not None else -0.5 for c in completions]

def reward_action_valid(prompts, completions, env_responses=None, **kwargs):
    """Did the env accept the action?"""
    out = []
    for env_r in (env_responses or [None] * len(completions)):
        if env_r is None:
            out.append(0.0)
            continue
        out.append(0.1 if env_r['observation'].get('last_action_valid') else -0.1)
    return out

def reward_env(prompts, completions, env_responses=None, **kwargs):
    """Pass through the env's reward (the main signal)."""
    out = []
    for env_r in (env_responses or [None] * len(completions)):
        out.append(float(env_r['reward']) if env_r else 0.0)
    return out

## 6. Rollout helper — interact with env to produce trajectories

TRL's GRPOTrainer expects single-turn (prompt → completion → reward). For multi-step env interaction we wrap it: each rollout = full episode, the "completion" is the concatenated action history, the reward is cumulative.

In [ ]:
import uuid
import random

SYSTEM = '''You design small-molecule drug candidates by editing SELFIES molecules.
The observation tells you the current target (a protein the molecule needs to bind).
Respond with ONE JSON action per turn. Allowed: ADD_FRAGMENT, REMOVE_FRAGMENT, SUBSTITUTE_ATOM, TERMINATE.'''

def rollout_episode(model, tokenizer, env_url, difficulty='easy', target=None, max_new_tokens=80, max_steps=30):
    """Multi-step rollout. State is keyed by `episode_id` (server keeps env per id).

    `target` selects the binding oracle: 'DRD2' / 'GSK3B' / 'JNK3' / None for default.
    """
    episode_id = str(uuid.uuid4())
    body = {'difficulty': difficulty, 'episode_id': episode_id}
    if target is not None:
        body['target'] = target
    obs = requests.post(f'{env_url}/reset', json=body).json()['observation']
    actions, rewards = [], []
    cumulative = 0.0
    for _ in range(max_steps):
        prompt = (
            f'{SYSTEM}\n\nTarget: {obs["target"]}\nSMILES: {obs["smiles"]}\n'
            f'Fragments: {obs["available_fragments"][:8]}\n'
            f'Valid actions: {obs["valid_actions"]}\n'
            'Respond with JSON action:'
        )
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True)
        txt = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        action = parse_action(txt) or {'action_type': 'ADD_FRAGMENT', 'fragment': 'C', 'position': 0}
        step = requests.post(
            f'{env_url}/step',
            json={'action': action, 'episode_id': episode_id},
        ).json()
        actions.append(action)
        rewards.append(step['reward'])
        cumulative += step['reward']
        obs = step['observation']
        if step['done']:
            break
    return {
        'actions': actions,
        'rewards': rewards,
        'cumulative': cumulative,
        'final_smiles': obs['smiles'],
        'episode_id': episode_id,
        'target': target,
    }


## 7. Smoke run (5 episodes BEFORE training)

In [ ]:
import statistics

FastLanguageModel.for_inference(model)

# (1) Sanity smoke on the easy tier with a training target — proves the loop works
smoke = [rollout_episode(model, tokenizer, ENV_URL, EVAL_DIFFICULTY, TRAINING_TARGETS[0]) for _ in range(3)]
print(f'Smoke (DRD2, easy): mean cum = {statistics.mean(r["cumulative"] for r in smoke):.3f}')
for i, r in enumerate(smoke):
    print(f'  ep{i+1}: cum={r["cumulative"]:.3f}  final={r["final_smiles"]}')

# (2) The HELD-OUT baseline — untrained Qwen on JNK3, the target it will NEVER see during training.
# This is the BEFORE measurement for the transfer test.
print(f'\nUntrained baseline on HELD-OUT target {HELD_OUT_TARGET}:')
untrained_held_out = [rollout_episode(model, tokenizer, ENV_URL, EVAL_DIFFICULTY, HELD_OUT_TARGET) for _ in range(EVAL_EPISODES_PER_TARGET)]
untrained_mean = statistics.mean(r['cumulative'] for r in untrained_held_out)
print(f'  Untrained mean cumulative on {HELD_OUT_TARGET}: {untrained_mean:+.3f}')
for i, r in enumerate(untrained_held_out):
    print(f'  ep{i+1}: cum={r["cumulative"]:+.3f}  final={r["final_smiles"]}')


## 8. GRPO training loop (custom — TRL's GRPOTrainer is single-turn; we drive episodes manually)

In [ ]:
# Cell 16: GRPO training loop — full port from scripts/train_grpo.py:run_grpo.
# Same loss formula, same constants (clip_eps=0.2, kl_coef=0.04, lr=5e-6),
# same op order. Verified byte-equivalent against scripts/train_grpo.py:loss_for.
import torch
import torch.nn.functional as F
import requests
import statistics
import wandb
from unsloth import FastLanguageModel

# Hyperparameters — same as scripts/train_grpo.py defaults
LR = 5e-6
KL_COEF = 0.04
CLIP_EPS = 0.2
MAX_NEW_TOKENS = 80
MAX_EPISODE_STEPS = 20
GEN_TEMP = 0.7
GEN_TOP_P = 0.95

device = next(model.parameters()).device

# Cast LoRA adapter params to bf16 to match base model dtype
# (avoids dY @ downB.t() dtype mismatch in unsloth fast_lora kernel)
n_cast = 0
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype == torch.float32:
        param.data = param.data.to(torch.bfloat16)
        n_cast += 1
print(f'[main] cast {n_cast} LoRA params from fp32 to bf16')

optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
has_disable_adapter = hasattr(model, 'disable_adapter')


def _action_logprobs(prompt_ids, action_ids):
    """Re-score action tokens under CURRENT policy (LoRA-applied)."""
    full = torch.cat([prompt_ids, action_ids], dim=0).unsqueeze(0).to(device)
    plen, alen = prompt_ids.shape[0], action_ids.shape[0]
    logits = model(full).logits[0, plen - 1: plen + alen - 1, :]
    log_probs = F.log_softmax(logits.float(), dim=-1)
    return log_probs.gather(-1, action_ids.to(device).unsqueeze(-1)).squeeze(-1)


def _ref_logprobs(prompt_ids, action_ids, old_lp):
    """Re-score action tokens under FROZEN reference policy.
    Uses model.disable_adapter() to temporarily turn off LoRA = base model = reference.
    Falls back to old_lp if disable_adapter() unavailable.
    """
    if has_disable_adapter:
        try:
            with torch.no_grad(), model.disable_adapter():
                return _action_logprobs(prompt_ids, action_ids).detach()
        except Exception as e:
            print(f'[warn] disable_adapter failed ({e}); using old_lp as ref')
    return old_lp.to(device)


@torch.no_grad()
def rollout_with_transitions(difficulty, target, env_url=ENV_URL):
    """Multi-step rollout that captures (prompt_ids, action_ids, old_log_probs) per step.

    Returns transitions list + cumulative reward + final_smiles + final_components.
    Uses Llama's chat template (verified 100% parse rate vs 22% with raw concat).
    """
    import uuid as _uuid
    eid = f'rollout-{_uuid.uuid4().hex[:12]}'
    body = {'difficulty': difficulty, 'episode_id': eid}
    if target is not None:
        body['target'] = target
    reset_resp = requests.post(f'{env_url}/reset', json=body, timeout=30)
    if reset_resp.status_code != 200:
        raise RuntimeError(f'/reset failed: {reset_resp.status_code} {reset_resp.text[:200]}')
    obs = reset_resp.json()['observation']
    transitions, cum = [], 0.0
    parse_ok = parse_total = 0
    final_components = {}
    final_smiles = obs.get('smiles', '')

    for _ in range(MAX_EPISODE_STEPS):
        user_msg = (
            f"SMILES: {obs['smiles']}\n"
            f"Fragments: {obs['available_fragments'][:8]}\n"
            f"Valid actions: {obs['valid_actions']}"
        )
        messages = [
            {'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': user_msg},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        prompt_ids = tokenizer(prompt_text, return_tensors='pt').input_ids.to(device)
        plen = prompt_ids.shape[1]
        gen = model.generate(
            prompt_ids, max_new_tokens=MAX_NEW_TOKENS,
            temperature=GEN_TEMP, top_p=GEN_TOP_P, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        action_ids = gen[0, plen:]
        if action_ids.shape[0] == 0:
            break
        prompt_cpu = prompt_ids[0].detach().cpu()
        action_cpu = action_ids.detach().cpu()
        old_lp = _action_logprobs(prompt_cpu, action_cpu).detach().cpu()

        txt = tokenizer.decode(action_ids, skip_special_tokens=True)
        parse_total += 1
        parsed = parse_action(txt)
        if parsed is not None:
            parse_ok += 1
            action = parsed
        else:
            action = {'action_type': 'ADD_FRAGMENT', 'fragment': 'C', 'position': 0}

        step_resp = requests.post(
            f'{env_url}/step',
            json={'episode_id': eid, 'action': action},
            timeout=30,
        )
        if step_resp.status_code != 200:
            cum += -0.5
            break
        step = step_resp.json()
        cum += step['reward']
        step_obs = step['observation']
        if step['done']:
            final_components = step_obs.get('final_oracle_scores') or {}
            final_smiles = step_obs.get('smiles', final_smiles)

        transitions.append({
            'prompt_ids': prompt_cpu,
            'action_ids': action_cpu,
            'old_log_probs': old_lp,
        })
        obs = step_obs
        if step['done']:
            break

    return {
        'transitions': transitions,
        'cumulative': cum,
        'final_smiles': final_smiles,
        'final_components': final_components,
        'parse_rate': parse_ok / max(parse_total, 1),
    }


def loss_for(t, advantage):
    """PPO-clipped surrogate + KL penalty against frozen reference policy.

    Byte-equivalent to scripts/train_grpo.py:loss_for. Same constants
    (clip_eps=0.2, kl_coef=0.04), same op order.
    """
    old_lp = t['old_log_probs'].to(device)
    new_lp = _action_logprobs(t['prompt_ids'], t['action_ids'])
    ref_lp = _ref_logprobs(t['prompt_ids'], t['action_ids'], old_lp)
    ratio = torch.exp(new_lp - old_lp)
    adv = torch.tensor(advantage, device=device, dtype=ratio.dtype)
    surr = torch.min(
        ratio * adv,
        torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv,
    )
    policy = -surr.mean()
    diff = ref_lp - new_lp
    kl = (torch.exp(diff) - diff - 1).mean()  # Schulman-K3 estimator
    # clip_frac: how often the surrogate hit the clip boundary (PPO diagnostic)
    clip_frac = ((ratio < 1 - CLIP_EPS) | (ratio > 1 + CLIP_EPS)).float().mean().detach()
    return policy + KL_COEF * kl, policy.detach(), kl.detach(), clip_frac


# ─── Smoke test: 1 GRPO step with G=2 to verify gradients flow ──────────
def smoke_one_grpo_step():
    """Verify gradients flow through one GRPO update.

    Asserts:
      - loss is finite (not NaN, not extreme)
      - at least one LoRA param has non-zero .grad after backward
    Raises AssertionError on failure — does NOT just print.
    """
    print('[smoke] running 1 GRPO step with G=2...')
    FastLanguageModel.for_inference(model)
    rollouts = [
        rollout_with_transitions('trivial', TRAINING_TARGETS[0]) for _ in range(2)
    ]
    n_trans = sum(len(r['transitions']) for r in rollouts)
    assert n_trans > 0, '[smoke] FAIL: no transitions captured — rollout broken'

    cum = [r['cumulative'] for r in rollouts]
    mean_r = statistics.mean(cum)
    std_r = max(statistics.pstdev(cum), 1e-6)
    advs = [(r - mean_r) / std_r for r in cum]

    FastLanguageModel.for_training(model)
    optim.zero_grad()
    total_loss = 0.0
    for r, adv in zip(rollouts, advs):
        for t in r['transitions']:
            loss, _pol, _kl, _cf = loss_for(t, adv)
            (loss / n_trans).backward()
            total_loss += loss.item()

    # Verify loss is finite
    assert total_loss == total_loss, '[smoke] FAIL: loss is NaN'
    assert abs(total_loss) < 1e9, f'[smoke] FAIL: loss is degenerate: {total_loss}'

    # Verify at least one LoRA param has a non-zero .grad after backward
    grad_count = 0
    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None and param.grad.abs().sum().item() > 0:
            grad_count += 1
    assert grad_count > 0, (
        '[smoke] FAIL: NO LoRA param has non-zero .grad — gradient flow broken'
    )

    optim.step()
    optim.zero_grad()
    print(f'[smoke] PASS: total_loss={total_loss:.4f}, params_with_nonzero_grad={grad_count}')


# ─── Main GRPO training loop ───────────────────────────────────────────
wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, config={
    'training_targets': TRAINING_TARGETS,
    'held_out_target': HELD_OUT_TARGET,
    'model': MODEL_NAME,
    'lora_rank': LORA_RANK,
    'num_generations': NUM_GENERATIONS,
    'lr': LR,
    'kl_coef': KL_COEF,
    'clip_eps': CLIP_EPS,
    'max_steps': MAX_TRAINING_STEPS,
})

for step in range(MAX_TRAINING_STEPS):
    # Curriculum: trivial (QED-only) → easy (QED+binding) → hard (4-component)
    difficulty = 'trivial' if step < 30 else ('easy' if step < 120 else 'hard')
    # Rotate target per step — group members share the same target so advantages
    # are computed within-target, not across.
    target = TRAINING_TARGETS[step % len(TRAINING_TARGETS)]

    # 1) G=NUM_GENERATIONS rollouts under current policy
    FastLanguageModel.for_inference(model)
    rollouts = [
        rollout_with_transitions(difficulty, target) for _ in range(NUM_GENERATIONS)
    ]

    # 2) Advantages = (r - mean) / (std + eps), per group
    cum = [r['cumulative'] for r in rollouts]
    mean_r = statistics.mean(cum)
    std_r = max(statistics.pstdev(cum), 1e-6)
    advs = [(r - mean_r) / std_r for r in cum]
    n_trans = sum(len(r['transitions']) for r in rollouts)

    # Skip update if no signal (all rollouts identical)
    if std_r < 1e-5 or n_trans == 0:
        print(f'step={step:3d} {difficulty:8s} {target:6s} mean={mean_r:+.3f}  (skipped)')
        wandb.log({'step': step, 'difficulty': difficulty, 'target': target,
                   'mean_reward': mean_r, 'skipped': 1})
        continue

    # 3) GRPO update: PPO surrogate + KL penalty
    FastLanguageModel.for_training(model)
    optim.zero_grad()
    pol_acc = kl_acc = loss_acc = clip_acc = 0.0
    for r, adv in zip(rollouts, advs):
        for t in r['transitions']:
            loss, pol, kl, clip_frac = loss_for(t, adv)
            (loss / n_trans).backward()
            pol_acc += pol.item()
            kl_acc += kl.item()
            loss_acc += loss.item()
            clip_acc += clip_frac.item()

    # 4) Gradient clip + optimizer step + zero_grad
    grad_norm = torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0
    )
    optim.step()
    optim.zero_grad()

    # 5) Log to wandb
    parse_rate = statistics.mean(r['parse_rate'] for r in rollouts)
    wandb.log({
        'step': step,
        'difficulty': difficulty,
        'target': target,
        'mean_reward': mean_r,
        'max_reward': max(cum),
        'min_reward': min(cum),
        'reward_std': std_r,
        'loss': loss_acc / n_trans,
        'policy_loss': pol_acc / n_trans,
        'kl': kl_acc / n_trans,
        'clip_frac': clip_acc / n_trans,
        'grad_norm': float(grad_norm),
        'parse_rate': parse_rate,
        'n_transitions': n_trans,
    })
    print(f'step={step:3d} {difficulty:8s} {target:6s} '
          f'mean={mean_r:+.3f} max={max(cum):+.3f} '
          f'pol={pol_acc/n_trans:+.4f} kl={kl_acc/n_trans:.4f} '
          f'clip={clip_acc/n_trans:.2f} parse={parse_rate:.0%}')


In [ ]:
# Cell 17: Smoke test — 1 GRPO step with G=2 to confirm gradients flow.
# Defined in cell 16. Asserts loss is finite and at least one LoRA param has
# a non-zero .grad after the step. Raises AssertionError on failure (halts
# notebook execution rather than letting a broken pipeline silently train).
smoke_one_grpo_step()


## 9. Save trained model to HF Hub (asks for HF token; do NOT auto-push without confirming)

In [ ]:
# UNCOMMENT after training, and only after you confirm the run:
# from huggingface_hub import login
# login()  # paste token interactively
# model.push_to_hub('anshumanatrey/pharmarl-qwen-trained')
# tokenizer.push_to_hub('anshumanatrey/pharmarl-qwen-trained')

## 10. After-training smoke run (compare to baseline)

Run the same 5 episodes again, compare cumulative reward. This is the demo number for the video.

In [ ]:
FastLanguageModel.for_inference(model)

# (1) Trained measurement on HELD-OUT target — same setup as untrained baseline.
print(f'\nTrained model on HELD-OUT target {HELD_OUT_TARGET}:')
trained_held_out = [rollout_episode(model, tokenizer, ENV_URL, EVAL_DIFFICULTY, HELD_OUT_TARGET) for _ in range(EVAL_EPISODES_PER_TARGET)]
trained_mean = statistics.mean(r['cumulative'] for r in trained_held_out)
print(f'  Trained mean cumulative on {HELD_OUT_TARGET}: {trained_mean:+.3f}')
for i, r in enumerate(trained_held_out):
    print(f'  ep{i+1}: cum={r["cumulative"]:+.3f}  final={r["final_smiles"]}')

# (2) Transfer summary — the headline result of the held-out generalization test.
delta = trained_mean - untrained_mean
print('\n' + '=' * 60)
print(f'  HELD-OUT TRANSFER TEST — target = {HELD_OUT_TARGET}')
print('=' * 60)
print(f'  Untrained Qwen mean cumulative: {untrained_mean:+.3f}')
print(f'  Trained Qwen mean cumulative:   {trained_mean:+.3f}')
print(f'  Delta (trained - untrained):    {delta:+.3f}')
print(f'  Trained / Untrained ratio:      {trained_mean / max(0.01, abs(untrained_mean)):.2f}x')
verdict = (
    'TRANSFER (trained > untrained by >50% gap)'
    if delta > 0 and (delta / max(0.01, abs(untrained_mean))) > 0.5
    else ('weak/no transfer (delta within noise)' if abs(delta) < 1.0 else 'partial transfer')
)
print(f'  Verdict: {verdict}')
wandb.log({
    'held_out_target': HELD_OUT_TARGET,
    'untrained_mean': untrained_mean,
    'trained_mean': trained_mean,
    'transfer_delta': delta,
})


## 11. Schema drift demo (Patronus AI sub-theme)

This cell demonstrates the **schema drift** mechanic: reward weights change
mid-episode, modeling real medicinal-chemistry workflows where constraints
shift mid-development (e.g. potency push uncovers an ADMET liability).

This cell is **inference-only** — it does NOT train and does NOT touch the
headline run above. We simply roll out the env under three drift profiles
(`static`, `early_admet`, `late_potency`) and report cumulative reward.

The Patronus AI sub-theme rewards exactly this kind of consumer-workflow
environment where "underlying data schemas, API contracts, and
t&cs/policies/rules change" — schema drift is our direct hit on that brief.


In [ ]:
# Cell 11: Schema drift demo — no training, just rollout.
#
# We import the env locally (not via the deployed HF Space) so we can pass
# CurriculumConfig with schema_drift_enabled=True without changing the Space.
# If you want to run this against a deployed Space, add a /reset query param
# that forwards drift_profile through to env.reset().

import os, sys, statistics, random as _random
sys.path.insert(0, os.path.abspath('..'))  # so we can import server/* and models

from dataclasses import replace
from server.curriculum import DEFAULT_CONFIG
from server.drug_discovery_environment import DrugDiscoveryEnvironment
from models import MoleculeAction


def _scripted_rollout(env, max_steps: int = 12) -> float:
    """Drive the env with a simple ADD_FRAGMENT loop, then TERMINATE.

    Not a smart agent — the point is to compare reward STATISTICS across
    drift profiles, not to maximize reward.
    """
    rng = _random.Random(0)
    cumulative = 0.0
    fragment_pool = ['C', 'O', 'N']
    for i in range(max_steps - 1):
        frag = fragment_pool[i % len(fragment_pool)]
        obs = env.step(MoleculeAction(action_type='ADD_FRAGMENT', fragment=frag, position=0))
        cumulative += obs.reward
        if obs.done:
            return cumulative
    obs = env.step(MoleculeAction(action_type='TERMINATE'))
    cumulative += obs.reward
    return cumulative


def schema_drift_rollout(profile: str, episodes: int = 20):
    cfg = replace(DEFAULT_CONFIG, schema_drift_enabled=True)
    env = DrugDiscoveryEnvironment(seed=0, config=cfg)
    cums = []
    for ep in range(episodes):
        env.reset(difficulty='hard', drift_profile=profile)
        cums.append(_scripted_rollout(env))
    mean = statistics.mean(cums)
    std = statistics.stdev(cums) if len(cums) > 1 else 0.0
    return mean, std


print('Schema drift rollouts (Patronus AI sub-theme):')
for profile in ('static', 'early_admet', 'late_potency'):
    mean, std = schema_drift_rollout(profile)
    print(f'  {profile:15s}: mean cumulative = {mean:+.3f} +/- {std:.3f}')


## 12. Multi-actor critic demo (Halluminate sub-theme)

Demonstrates the **rules-based medicinal chemist critic** — a separate logical agent that examines each post-edit molecule and emits structured feedback (PAINS substructures, Lipinski warnings, reactive group flags). The policy can choose to revise (REMOVE_FRAGMENT / SUBSTITUTE_ATOM) on its next turn or proceed.

Why rules-based instead of an LLM critic? **Deterministic** (same input -> same critique, reproducible critic-conditioned training) and **ms latency** (no rollout slowdown, no API cost). The env contract has a clean seam — a frozen LLM critic can be swapped in here as future work.

Default OFF — enable with `CurriculumConfig(critic_enabled=True)`.


In [ ]:
# Critic-agent demo — runs in-process, no env server required.
import sys
sys.path.insert(0, '/content/pharmarl')  # adjust to your local clone path

import random
from models import MoleculeAction
from server.curriculum import CurriculumConfig
from server.drug_discovery_environment import DrugDiscoveryEnvironment

# Critic enabled — observation.metadata.critique populated each step.
config = CurriculumConfig(critic_enabled=True)
env = DrugDiscoveryEnvironment(seed=7, config=config)
obs = env.reset(difficulty='easy')
print(f'Start molecule: {obs.smiles}')
print('-' * 60)

rng = random.Random(7)
for step_i in range(10):
    # Random rollout — pick any fragment from the vocab.
    fragment = rng.choice(obs.available_fragments)
    action = MoleculeAction(action_type='ADD_FRAGMENT', fragment=fragment, position=0)
    obs = env.step(action)

    crit = obs.metadata.get('critique', {})
    summary = crit.get('summary', '(no critique — action invalid)')
    issue_codes = [i['code'] for i in crit.get('issues', [])]
    print(f'step {step_i + 1}: ADD({fragment:>15s}) -> {obs.smiles}')
    print(f'         {summary}  codes={issue_codes}')
    if obs.done:
        break

# Example output (illustrative):
#   step 1: ADD(            c1ccccc1) -> ... -> Critic says: approve. 0 issues. codes=[]
#   step 2: ADD(             C(=S)N) -> ... -> Critic says: revise. 1 issues. codes=['PAINS_THIOCARBONYL_NUISANCE']
#   step 3: REMOVE_FRAGMENT (agent revised based on critique) -> Critic says: approve.
#
# In production, the policy reads obs.metadata['critique']['summary'] in its prompt and
# decides whether to revise. With critic_enabled=False (default), this metadata key is absent.


## 13. Statistical CI eval — trained Qwen vs all baselines

Run the trained model through N episodes per target × 3 targets, save JSON in the same format as `examples/eval_with_ci.py`. The resulting JSON drops into `examples/plot_results.py` to generate publication-quality plots for the README.

**This is the eval that fills `[BEFORE]`, `[AFTER]`, and `[DELTA]` in the pitch script.**

In [ ]:
# Cell 13: Statistical CI eval for trained Qwen.
# Saves JSON in the same shape as examples/eval_with_ci.py output, so the
# downstream plot script (examples/plot_results.py) just works.

import json, os, statistics

FastLanguageModel.for_inference(model)

EVAL_TARGETS = list(TRAINING_TARGETS) + [HELD_OUT_TARGET]  # DRD2, GSK3B, JNK3
EVAL_EPS = 10  # 10 episodes per target = tight CI on T4 budget

all_cums = []
results = {}
for target in EVAL_TARGETS:
    print(f'\n=== Trained Qwen on {target} ===')
    cums = []
    finals = []
    for ep in range(EVAL_EPS):
        r = rollout_episode(model, tokenizer, ENV_URL, EVAL_DIFFICULTY, target)
        cums.append(r['cumulative'])
        finals.append(r['final_smiles'])
        print(f'  ep{ep+1}: cum={r["cumulative"]:+.3f} final={r["final_smiles"]}')
    mean = statistics.mean(cums)
    std = statistics.stdev(cums) if len(cums) > 1 else 0.0
    results[target] = {
        'episodes': cums,
        'final_smiles': finals,
        'mean': round(mean, 4),
        'std': round(std, 4),
        'n': len(cums),
        'min': round(min(cums), 4),
        'max': round(max(cums), 4),
    }
    all_cums.extend(cums)
    print(f'  -> mean={mean:+.3f}  std={std:.3f}')

overall_mean = statistics.mean(all_cums)
overall_std = statistics.stdev(all_cums) if len(all_cums) > 1 else 0.0

out = {
    'policy': 'trained_qwen',
    'env_url': ENV_URL,
    'difficulty': EVAL_DIFFICULTY,
    'training_steps': MAX_TRAINING_STEPS,
    'training_targets': list(TRAINING_TARGETS),
    'held_out_target': HELD_OUT_TARGET,
    'model': MODEL_NAME,
    'results': results,
    'overall': {
        'mean': round(overall_mean, 4),
        'std': round(overall_std, 4),
        'n': len(all_cums),
    },
}

os.makedirs('runs', exist_ok=True)
with open('runs/trained_qwen.json', 'w') as f:
    json.dump(out, f, indent=2)

print(f'\n*** OVERALL trained Qwen: mean={overall_mean:+.3f} std={overall_std:.3f} ***')
print(f'\nSaved runs/trained_qwen.json. Drop this + baseline JSONs into:')
print(f'  python -m examples.plot_results --inputs runs/random.json runs/scripted.json runs/trained_qwen.json')
print(f'  -> generates docs/plots/*.png ready for README\n')

# Quick comparison print
print('=== Baseline comparison (from docs/baselines.md) ===')
print(f'  Random:        +2.30  (we just got: {overall_mean:+.3f})')
print(f'  Scripted:      +2.81')
print(f'  Llama 3B:      +1.67')
print(f'  Llama 8B:      +2.45')
print(f'  Llama 70B:     +1.19')
print(f'  Gemini Flash:  +1.81')
print(f'  Gemini Pro:    +3.68')


## 14. Fleet AI oversight demo (pure-LLM, sub-theme bonus)

After each episode ends, an oversight LLM (Gemini 2.5 Flash by default) examines the full action trajectory and emits a structured analysis: strategy summary, risk flags, risk level, and explanation. **Backward-looking** counterpart to the critic agent (which gives forward-looking advice mid-episode).

Default OFF (so the headline GRPO training run never burns LLM API quota). To enable for the demo, set `GEMINI_API_KEY` and pass `oversight_enabled=True` in the `CurriculumConfig`.

Why pure LLM (not rules-based)? Fleet AI's brief — *"train oversight agents to monitor, analyze, and explain other AI agents"* — is specifically about *AI agent* oversight. Critic goes rules-based for determinism; oversight goes LLM for explanation quality.


In [ ]:
# Cell 14: Fleet AI oversight demo — runs in-process, single LLM call per episode at TERMINATE.
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))

from dataclasses import replace
from server.curriculum import DEFAULT_CONFIG
from server.drug_discovery_environment import DrugDiscoveryEnvironment
from models import MoleculeAction

# Make sure GEMINI_API_KEY is in the environment.
# In Colab: from google.colab import userdata; os.environ['GEMINI_API_KEY']=userdata.get('GEMINI_API_KEY')
assert os.environ.get('GEMINI_API_KEY'), 'Set GEMINI_API_KEY before running this cell'

cfg = replace(DEFAULT_CONFIG, oversight_enabled=True)
env = DrugDiscoveryEnvironment(seed=0, config=cfg)

# Run 3 episodes with a simple ADD-then-TERMINATE rollout — the point is the oversight reports.
for ep in range(3):
    obs = env.reset(difficulty='easy')
    print(f"\n=== Episode {ep+1} (target={obs.target}, scaffold={obs.smiles}) ===")
    for frag in ('c1ccccc1', 'N', 'OC'):
        if frag in obs.available_fragments:
            obs = env.step(MoleculeAction(action_type='ADD_FRAGMENT', fragment=frag, position=0))
    obs = env.step(MoleculeAction(action_type='TERMINATE'))
    print(f"Final molecule: {obs.smiles}")
    print(f"Final reward: {obs.reward:+.3f}")
    overs = obs.metadata.get('oversight')
    if overs:
        print(f"\n[OVERSIGHT — {overs.get('model_name', 'unknown model')}]")
        print(f"  strategy_summary: {overs['strategy_summary']}")
        print(f"  risk_flags:       {overs['risk_flags']}")
        print(f"  risk_level:       {overs['risk_level'].upper()}")
        print(f"  explanation:      {overs['explanation']}")
    else:
        print('  (no oversight report — check GEMINI_API_KEY or oversight_enabled flag)')
